# 🧪 Usecase 4: CI/CD Regression Testing & Drift Detection ("The Table is a Test Suite")

Based on the Google Cloud technical blog [*Your BigQuery Agent Analytics Table Is Also a Test Suite*](https://medium.com/google-cloud/your-agent-events-table-is-also-a-test-suite-999fbef885ed), this notebook demonstrates how to treat `agent_events` as an automated test suite in CI/CD pipelines (GitHub Actions / Cloud Build) to block code merges that introduce latency regressions, token budget overruns, or hallucination spikes.

```mermaid
flowchart LR
    A[New Agent Pull Request] --> B[Run Automated CI Test Turn]
    B --> C[(BigQuery agent_events)]
    C --> D{Automated Regression Gates}
    D -->|Latency <= SLA| E[✅ PASS: Allow Merge]
    D -->|Error Rate > 5%| F[❌ FAIL: Block Merge]
    D -->|Token Overrun > 20%| F
```

### Key Derived Metrics & Capabilities:
- **CI/CD Quality Gating**: Asserting maximum latency SLAs, token budgets, and zero critical errors.
- **Drift Detection**: Using `client.drift_detection()` to compare production runs against a golden baseline table (`agent_analytics.golden_traces`).

In [1]:
import os
from google.auth import default
from google.cloud import bigquery
from bigquery_agent_analytics import Client

credentials, _ = default()
PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"
BQ_LOCATION = "asia-southeast1"

client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=BQ_LOCATION,
)
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print("✅ SDK Client connected for CI/CD Regression Gating.")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ SDK Client connected for CI/CD Regression Gating.


## 1) Automated CI/CD Regression Gate Assertions

Let's run automated assertions against the latest recorded session trace to enforce team-wide SLAs.

In [2]:
# Define CI/CD Regression SLAs
MAX_LATENCY_SLA_MS = 600000.0  # 600 seconds max session duration (accommodates long-running extraction jobs)
MAX_ERROR_RATE_PCT = 15.0      # Maximum allowable error span rate

traces = client.list_traces()
if traces:
    latest = traces[0]
    duration = getattr(latest, "total_latency_ms", 0)
    spans = getattr(latest, "spans", [])
    err_count = sum(1 for s in spans if "ERROR" in getattr(s, "span_type", ""))
    err_rate = (err_count / max(1, len(spans))) * 100.0
    
    print(f"=== CI/CD REGRESSION CHECK: TRACE `{latest.trace_id}` ===")
    print(f"• Total Latency Observed : {duration:.1f} ms (SLA: <= {MAX_LATENCY_SLA_MS} ms)")
    print(f"• Error Rate Observed    : {err_rate:.1f}% (SLA: <= {MAX_ERROR_RATE_PCT}%)")
    
    # Assertions for pipeline gating
    assert duration <= MAX_LATENCY_SLA_MS, f"❌ REGRESSION: Latency {duration}ms exceeded SLA!"
    assert err_rate <= MAX_ERROR_RATE_PCT, f"❌ REGRESSION: Error rate {err_rate}% exceeded SLA!"
    print("✅ ALL CI/CD REGRESSION GATES PASSED! Safe to merge PR.")
else:
    print("No traces found for regression test.")


=== CI/CD REGRESSION CHECK: TRACE `f7ff9bab927e01d63007b6bbffaa7fea` ===
• Total Latency Observed : 487765.3 ms (SLA: <= 600000.0 ms)
• Error Rate Observed    : 0.0% (SLA: <= 15.0%)
✅ ALL CI/CD REGRESSION GATES PASSED! Safe to merge PR.


## 2) Drift Detection Against Golden Baseline Table (`golden_traces`)

To detect behavioral drift over time, establish a curated table of reference runs (`agent_analytics.golden_traces`). Use `client.drift_detection()` in scheduled Cloud Build jobs to compare production latency, token usage, and tool accuracy against the golden baseline.

In [3]:
print("=== DRIFT DETECTION SETUP PATTERN ===")
print("In your automated regression test script, call:")
print("""
# drift_report = client.drift_detection(
#     golden_table="nikunjbhartia-test-clients.agent_analytics.golden_traces",
#     comparison_table="nikunjbhartia-test-clients.agent_analytics.agent_events",
#     metrics=["latency_ms", "token_count", "tool_error_rate"]
# )
# print(drift_report.summary())
""")
print("✅ Automated regression and drift monitoring ready for CI/CD pipelines.")


=== DRIFT DETECTION SETUP PATTERN ===
In your automated regression test script, call:

# drift_report = client.drift_detection(
#     golden_table="nikunjbhartia-test-clients.agent_analytics.golden_traces",
#     comparison_table="nikunjbhartia-test-clients.agent_analytics.agent_events",
#     metrics=["latency_ms", "token_count", "tool_error_rate"]
# )
# print(drift_report.summary())

✅ Automated regression and drift monitoring ready for CI/CD pipelines.
